In [2]:
import os
os.chdir(r'C:\Users\johnpaul\fraudguard-africa')
print(os.getcwd())

C:\Users\johnpaul\fraudguard-africa


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('data/PS_20174392719_1491204439457_log.csv')

print("Original Shape:", df.shape)

Original Shape: (6362620, 11)


In [4]:
# 1. Basic Balance Features
df['balance_diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balance_diff_dest'] = df['oldbalanceDest'] - df['newbalanceDest']

# 2. Transaction Amount vs Balance Ratio
df['amount_to_oldbalance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)  # +1 to avoid division by zero
df['amount_to_newbalance_ratio'] = df['amount'] / (df['newbalanceOrig'] + 1)

In [5]:
# 3. Time-based Features
df['hour'] = df['step'] % 24
df['is_night'] = df['hour'].isin([0,1,2,3,4,5,22,23]).astype(int)

# 4. Transaction Type Encoding
df = pd.get_dummies(df, columns=['type'], prefix='type', drop_first=True)

In [6]:
# 5. Customer Behavior Features (Important!)

# Number of transactions per customer (nameOrig)
orig_freq = df['nameOrig'].value_counts()
df['orig_transaction_freq'] = df['nameOrig'].map(orig_freq)

# 6. Fraud Risk Score (Rule-based simple feature)
df['high_risk_transaction'] = ((df['type_CASH_OUT'] == 1) & 
                               (df['amount'] > 100000)).astype(int)

print("New Shape after Feature Engineering:", df.shape)
print("\nNew Features Created:")
print([col for col in df.columns if col not in ['step', 'type', 'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud']])

# Save the engineered dataset (optional for now)
# df.to_csv('data/engineered_data.csv', index=False)

New Shape after Feature Engineering: (6362620, 22)

New Features Created:
['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig', 'balance_diff_dest', 'amount_to_oldbalance_ratio', 'amount_to_newbalance_ratio', 'hour', 'is_night', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'orig_transaction_freq', 'high_risk_transaction']
